# EEG Feature Optimization

This notebook evaluates whether removing redundant features and applying log-transformation to absolute band power improves motor imagery classification.

Four feature representations are compared using LDA and Leave-One-Subject-Out cross-validation to assess generalization across participants.

## 1. Data Loading and Preparation

Load the processed EEG dataset and retain only T1 (left-hand imagery) and T2 (right-hand imagery).

The original feature set contains 21 handcrafted EEG features extracted from channels C3, Cz, and C4.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path("../data/processed/eeg_features_all_subjects.csv")
df = pd.read_csv(data_path)

binary_df = df[df["event"].isin(["T1", "T2"])].copy()

metadata_columns = ["event", "subject_id", "recording_id"]
feature_columns = [col for col in binary_df.columns if col not in metadata_columns]

X = binary_df[feature_columns]
y = binary_df["event"].map({"T1": 0, "T2": 1})
groups = binary_df["subject_id"]

print("Feature matrix:", X.shape)
print("Subjects:", groups.nunique())
print("\nClass distribution:")
print(y.value_counts().sort_index())

Feature matrix: (225, 21)
Subjects: 5

Class distribution:
event
0    113
1    112
Name: count, dtype: int64


## 2. Feature Redundancy Reduction

Correlation analysis identified six highly redundant feature pairs.

RMS and relative beta power features are excluded from the reduced representation, leaving 15 features for comparison with the original 21-feature dataset.

In [2]:
features_to_drop = ["C3.._rms", "Cz.._rms", "C4.._rms", 
                    "C3.._relative_beta_power", "Cz.._relative_beta_power", "C4.._relative_beta_power"]
X_reduced = X.drop(columns=features_to_drop)

print("Original feature matrix:", X.shape)
print("Reduced feature matrix:", X_reduced.shape)

print("\nRemoved features:")
print(features_to_drop)

print("\nRemaining features:")
print(X_reduced.columns.tolist())

Original feature matrix: (225, 21)
Reduced feature matrix: (225, 15)

Removed features:
['C3.._rms', 'Cz.._rms', 'C4.._rms', 'C3.._relative_beta_power', 'Cz.._relative_beta_power', 'C4.._relative_beta_power']

Remaining features:
['C3.._mean', 'C3.._std', 'C3.._alpha_power', 'C3.._beta_power', 'C3.._relative_alpha_power', 'Cz.._mean', 'Cz.._std', 'Cz.._alpha_power', 'Cz.._beta_power', 'Cz.._relative_alpha_power', 'C4.._mean', 'C4.._std', 'C4.._alpha_power', 'C4.._beta_power', 'C4.._relative_alpha_power']


**Methodological note:** Relative beta power was removed because it is mathematically redundant with relative alpha power under the current feature definition. RMS features were excluded based on exploratory correlation analysis across the initial five subjects.

These feature choices are exploratory and should be predefined or selected using training data only during independent model evaluation.

## 3. Log-Transformed Feature Sets

Apply logarithmic transformation to the six absolute alpha and beta power features.

Both the original and reduced feature sets are retained, allowing us to compare the effects of feature reduction and log-transformation separately.

In [3]:
power_columns = [col for col in X.columns 
                 if col.endswith(("_alpha_power", "_beta_power")) and "relative" not in col]
epsilon = 1e-30

X_log = X.copy()
X_reduced_log = X_reduced.copy()

X_log[power_columns] = np.log(X_log[power_columns].clip(lower=epsilon))
X_reduced_log[power_columns] = np.log(X_reduced_log[power_columns].clip(lower=epsilon))

feature_sets = {"Original (21)": X, "Reduced (15)": X_reduced, "Log (21)": X_log, "Reduced + Log (15)": X_reduced_log}

print("Transformed features:", power_columns)

print("\nFeature sets:")
for name, features in feature_sets.items():
    print(f"{name}: {features.shape}")

Transformed features: ['C3.._alpha_power', 'C3.._beta_power', 'Cz.._alpha_power', 'Cz.._beta_power', 'C4.._alpha_power', 'C4.._beta_power']

Feature sets:
Original (21): (225, 21)
Reduced (15): (225, 15)
Log (21): (225, 21)
Reduced + Log (15): (225, 15)


## 4. LDA Cross-Validation Comparison

Evaluate four feature representations using LDA with Leave-One-Subject-Out cross-validation.

The same subject-level splits are used for all experiments. StandardScaler is fitted within each training fold to prevent data leakage.

Accuracy, balanced accuracy, F1-score, and Macro-F1 are compared to assess whether feature reduction or log-transformation improves cross-subject classification.

In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import LeaveOneGroupOut, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logo = LeaveOneGroupOut()
scoring = {"accuracy": "accuracy", 
           "balanced_accuracy": "balanced_accuracy", 
           "f1": "f1",
           "f1_macro": "f1_macro"}
results = []

for feature_set_name, X_features in feature_sets.items():
    model = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())

    cv_results = cross_validate(model, X_features, y, groups=groups, cv=logo, scoring=scoring)

    for metric in scoring:
        scores = cv_results[f"test_{metric}"]
        results.append({"Feature Set": feature_set_name,
                        "Metric": metric,
                        "Mean": scores.mean(),
                        "Std": scores.std()})
comparison_df = pd.DataFrame(results)

display(comparison_df.round(3))

,Feature Set,Metric,Mean,Std
0,Original (21),accuracy,0.556,0.064
1,Original (21),balanced_accuracy,0.558,0.064
2,Original (21),f1,0.556,0.099
3,Original (21),f1_macro,0.538,0.052
4,Reduced (15),accuracy,0.556,0.064
5,Reduced (15),balanced_accuracy,0.558,0.064
6,Reduced (15),f1,0.556,0.099
7,Reduced (15),f1_macro,0.538,0.052
8,Log (21),accuracy,0.538,0.082
9,Log (21),balanced_accuracy,0.537,0.083


### Feature Optimization Results

Reducing the feature set from 21 to 15 features preserved both balanced accuracy (**0.558**) and Macro-F1 (**0.538**).

Log-transformation did not improve classification performance.

The reduced feature set provides a simpler representation without an observed loss in cross-validation performance.

## 5. Subject-Level Performance

Compare balanced accuracy across individual subjects to examine whether feature reduction and log-transformation affect participants consistently.

In [5]:
from sklearn.model_selection import cross_val_score

subject_results = pd.DataFrame(index=sorted(groups.unique()))

for feature_set_name, X_features in feature_sets.items():
    model = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())

    scores = cross_val_score(model, X_features, y, groups=groups, cv=logo, scoring="balanced_accuracy")

    subject_results[feature_set_name] = scores

subject_results.index.name = "Subject"

display(subject_results.round(3))

,Original (21),Reduced (15),Log (21),Reduced + Log (15)
Subject,,,,
S001,0.488,0.488,0.578,0.578
S002,0.618,0.618,0.666,0.666
S003,0.516,0.516,0.532,0.532
S004,0.652,0.652,0.494,0.494
S005,0.518,0.518,0.417,0.417


### Subject-Level Results

Feature reduction produced identical balanced accuracy for all five subjects.

Log-transformation improved performance for S001–S003 but reduced it for S004 and S005, with the largest decrease observed for S004 (−0.158).

These results highlight subject-dependent effects of feature transformations and suggest that log-transformation does not consistently improve cross-subject generalization.

## 6. Regularized LDA

Feature reduction and log-transformation did not improve cross-subject classification.

We now investigate whether covariance regularization can improve LDA performance. Standard LDA is compared with automatic and fixed shrinkage using the reduced 15-feature dataset and identical Leave-One-Subject-Out splits.

In [6]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lda_models = {"Standard LDA": LinearDiscriminantAnalysis(),
              "LDA Auto Shrinkage": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
              "LDA Shrinkage 0.5": LinearDiscriminantAnalysis(solver="lsqr", shrinkage=0.5)}
regularization_results = []

for model_name, lda in lda_models.items():
    model = make_pipeline(StandardScaler(), lda)

    cv_results = cross_validate(model, X_reduced, y, groups=groups, cv=logo, scoring=scoring)

    for metric in scoring:
        scores = cv_results[f"test_{metric}"]

        regularization_results.append({"Model": model_name,
                                       "Metric": metric,
                                       "Mean": scores.mean(),
                                       "Std": scores.std()})
regularization_df = pd.DataFrame(regularization_results)

display(regularization_df.round(3))

,Model,Metric,Mean,Std
0,Standard LDA,accuracy,0.556,0.064
1,Standard LDA,balanced_accuracy,0.558,0.064
2,Standard LDA,f1,0.556,0.099
3,Standard LDA,f1_macro,0.538,0.052
4,LDA Auto Shrinkage,accuracy,0.524,0.070
5,LDA Auto Shrinkage,balanced_accuracy,0.523,0.077
6,LDA Auto Shrinkage,f1,0.572,0.073
7,LDA Auto Shrinkage,f1_macro,0.486,0.083
8,LDA Shrinkage 0.5,accuracy,0.502,0.065
9,LDA Shrinkage 0.5,balanced_accuracy,0.499,0.072


### Regularized LDA Results

Standard LDA achieved the highest balanced accuracy (**0.558**) and Macro-F1 (**0.538**).

Automatic shrinkage slightly increased the binary F1-score but reduced Macro-F1 and balanced accuracy, indicating no overall improvement across both classes.

Covariance regularization did not improve cross-subject classification in the evaluated configurations.

## Conclusion

Feature optimization and LDA regularization were evaluated using Leave-One-Subject-Out cross-validation.

Key findings:

- Reducing 21 features to 15 preserved classification performance.
- Log-transformation reduced mean balanced accuracy from **0.558 to 0.537**.
- Automatic and fixed LDA shrinkage did not improve performance.
- Standard LDA with 15 features remained the preferred configuration, achieving **0.558 balanced accuracy**.

These results suggest that the tested feature transformations and regularization methods are insufficient to substantially improve cross-subject classification.

**Next step:** Investigate Common Spatial Patterns (CSP) to extract spatial information directly from EEG signals and compare its performance with handcrafted features.